In [2]:
# === IMPORTS ===
import requests
import time
import pandas as pd
from dateutil import parser

# === CONFIGURATION ===
API_KEY = 'Fz8Uw2qYs8f7ndnPs03UI1Ya0mfGwQU0'
START_YEAR = 2014
END_YEAR = 2024   # set to 2024 for full crawl

CLIMATE_KEYWORDS = [

     # Major institutions & agreements (global in scope)
    "IPCC", "Intergovernmental Panel on Climate Change",
    "UNFCCC", "Paris Agreement", "COP", "un climate summit", "global climate summit",
    "climate accord", "climate pact", "un climate report", "u.n. climate report",
    "climate talks",
    
    # Energy transition & global policy
    "International Energy Agency", "IEA World Energy Outlook", "IEA Net Zero",
    "World Energy Outlook", "energy transition", "renewable energy",
    "carbon market", "carbon trading", "cap and trade",
    
    # Financial & regulatory (systemic impact)
    "SEC climate disclosure", "EPA climate rule",
    "European Union Green Deal", "EU climate law",
    "World Bank climate report", "IMF climate report",
    "OECD climate outlook", "NGFS climate scenario",
    "sustainability goals", "ESG regulation", "greenwashing"
]

VALID_SECTIONS = {"world", "business", "climate", "science"}


# === FUNCTION TO FETCH ARTICLES ===
def fetch_monthly_articles(year, month, api_key, max_retries=5):
    url = f"https://api.nytimes.com/svc/archive/v1/{year}/{month}.json"
    params = {'api-key': api_key}

    for attempt in range(max_retries):
        response = requests.get(url, params=params)
        if response.status_code == 200:
            try:
                return response.json()['response']['docs']
            except Exception as e:
                print(f"JSON parsing error for {year}-{month:02d}: {e}")
                return []
        elif response.status_code == 429:
            wait_time = 60 * (attempt + 1)
            print(f"Rate limit hit. Waiting {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            print(f"Failed for {year}-{month:02d} | Status: {response.status_code}")
            return []
    print(f"Gave up on {year}-{month:02d} after {max_retries} retries.")
    return []

# === FUNCTION TO FILTER ARTICLES ===
def is_climate_article(article):
    text = (
        (article.get("headline", {}) or {}).get("main", "") + " " +
        (article.get("snippet") or "") + " " +
        (article.get("lead_paragraph") or "")
    ).lower()

    headline = (article.get("headline", {}) or {}).get("main", "").lower()

    score = 0
    for kw in CLIMATE_KEYWORDS:
        if kw in headline:
            score += 2  # higher weight for headline matches
        elif kw in text:
            score += 1

    return score >= 2  # keep only if strong enough signal

# === FUNCTION TO EXTRACT RELEVANT DATA ===
def extract_relevant_data(article):
    pub_raw = article.get('pub_date', '')
    try:
        pub_datetime = parser.isoparse(pub_raw)
        pub_date = pub_datetime.strftime('%Y-%m-%d')
        pub_time = pub_datetime.strftime('%H:%M:%S')
    except:
        pub_date = pub_raw[:10]
        pub_time = ''

    return {
        'pub_date': pub_date,
        'pub_time': pub_time,
        'headline': (article.get("headline", {}) or {}).get("main", ""),
        'url': article.get("web_url", ""),
        'section': article.get("section_name", ""),
        'source': article.get("source", ""),
        'snippet': article.get("snippet", ""),
        'byline': (article.get("byline", {}) or {}).get("original", ""),
        'news_desk': article.get("news_desk", ""),
        'material_type': article.get("type_of_material", "")
    }

# === MAIN SCRAPER FUNCTION ===
def collect_climate_articles(api_key, start_year, end_year):
    all_articles = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            print(f"Fetching {year}-{month:02d}...")
            articles = fetch_monthly_articles(year, month, api_key)
            for article in articles:
                if is_climate_article(article):
                    section = (article.get("section_name") or "").lower()
                    if not VALID_SECTIONS or section in VALID_SECTIONS:
                        all_articles.append(extract_relevant_data(article))
            time.sleep(7)  # safe spacing to avoid 429s

    return pd.DataFrame(all_articles)

# === RUN SCRIPT ===
if __name__ == "__main__":
    df_climate_news = collect_climate_articles(API_KEY, START_YEAR, END_YEAR)
    df_climate_news.to_csv("nyt_climate_news_updated.csv", index=False)
    print(f"\n✅ Done! {len(df_climate_news)} articles saved to 'nyt_climate_news_updated.csv'")


Fetching 2014-01...
Fetching 2014-02...
Fetching 2014-03...
Fetching 2014-04...
Fetching 2014-05...
Fetching 2014-06...
Fetching 2014-07...
Fetching 2014-08...
Fetching 2014-09...
Fetching 2014-10...
Fetching 2014-11...
Fetching 2014-12...
Fetching 2015-01...
Fetching 2015-02...
Fetching 2015-03...
Fetching 2015-04...
Fetching 2015-05...
Fetching 2015-06...
Fetching 2015-07...
Fetching 2015-08...
Rate limit hit. Waiting 60 seconds...
Fetching 2015-09...
Fetching 2015-10...
Fetching 2015-11...
Fetching 2015-12...
Fetching 2016-01...
Fetching 2016-02...
Rate limit hit. Waiting 60 seconds...
Fetching 2016-03...
Fetching 2016-04...
Fetching 2016-05...
Fetching 2016-06...
Fetching 2016-07...
Fetching 2016-08...
Fetching 2016-09...
Fetching 2016-10...
Fetching 2016-11...
Fetching 2016-12...
Fetching 2017-01...
Rate limit hit. Waiting 60 seconds...
Fetching 2017-02...
Fetching 2017-03...
Fetching 2017-04...
Fetching 2017-05...
Fetching 2017-06...
Fetching 2017-07...
Rate limit hit. Waiting 60